# analysis_failure_modes_v1

目标：验证 `src.analysis` 可以把 regime/window/model delta 归纳为 failure mode 清单，并输出 `failure_modes.csv` 与中文 `failure_modes.md`。

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists() and (path / 'src').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.analysis import (
    analyze_model_effects_by_regime_and_window,
    build_failure_modes,
    build_regime_window_analysis_report,
    compare_models,
    compute_regime_metrics,
    compute_window_metrics,
    write_analysis_report,
    write_failure_modes_markdown,
)
from src.core import DatasetSnapshot

CARD = REPO_ROOT / 'research/analysis_failure_modes_v1'
REPORT_DIR = CARD / 'analysis_report'
CARD.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
{'card': str(CARD.relative_to(REPO_ROOT)), 'report_dir': str(REPORT_DIR.relative_to(REPO_ROOT))}

{'card': 'research/analysis_failure_modes_v1',
 'report_dir': 'research/analysis_failure_modes_v1/analysis_report'}

## 构造模型路径

构造一个 ridge baseline 和一个 graph/sequence 类候选模型。候选模型在 bull 段收益更高，但在 stress window 下相对收益、稳定性和回撤同时变差。

In [2]:
dates = list(pd.date_range('2026-01-01', periods=8, freq='D')) * 2
regimes = ['bull', 'bull', 'bull', 'neutral', 'stress', 'stress', 'stress', 'downtrend'] * 2
daily = pd.DataFrame({
    'date': dates,
    'model': ['ridge_mse'] * 8 + ['gcn_mse'] * 8,
    'regime': regimes,
    'nav': [
        1.00, 1.02, 1.04, 1.03, 1.01, 0.99, 1.00, 1.01,
        1.00, 1.04, 1.08, 1.07, 1.02, 0.96, 0.97, 0.98,
    ],
    'benchmark_nav': [1.0, 1.01, 1.02, 1.02, 1.00, 0.99, 1.00, 1.00] * 2,
    'position': [0.2, 0.3, 0.5, 0.1, 0.4, 0.2, 0.1, 0.2] * 2,
    'turnover': [0.0, 0.1, 0.2, 0.4, 0.3, 0.1, 0.1, 0.2] * 2,
})
daily.to_csv(CARD / 'daily_backtest.csv', index=False)
daily.head()

,date,model,regime,nav,benchmark_nav,position,turnover
0,2026-01-01,ridge_mse,bull,1.00,1.00,0.2,0.0
1,2026-01-02,ridge_mse,bull,1.02,1.01,0.3,0.1
2,2026-01-03,ridge_mse,bull,1.04,1.02,0.5,0.2
3,2026-01-04,ridge_mse,neutral,1.03,1.02,0.1,0.4
4,2026-01-05,ridge_mse,stress,1.01,1.00,0.4,0.3


## 运行 analysis report

通过已有 qlib/regime/window 分析接口生成标准 AnalysisReport。`failure_modes.csv` 由 `model_effect_deltas` 自动派生。

In [3]:
windows = {
    'early': ('2026-01-01', '2026-01-04'),
    'stress_window': ('2026-01-05', '2026-01-07'),
}
effect = analyze_model_effects_by_regime_and_window(daily, windows=windows, baseline_model='ridge_mse')
failure_modes = build_failure_modes(effect.model_effect_deltas, top_n=10)
report = build_regime_window_analysis_report(
    artifact_id='analysis_failure_modes_v1',
    effect_analysis=effect,
    model_comparison=compare_models(daily),
    regime_metrics=compute_regime_metrics(daily),
    window_metrics=compute_window_metrics(daily, windows=windows),
    summary={
        'status': 'analysis_failure_modes_completed',
        'note': '中文 review 入口：failure_modes 从 regime/window/model delta 派生。',
    },
)
dataset = DatasetSnapshot(
    name='failure_modes_demo_daily',
    universe=('SOXL', 'SOXX'),
    start='2026-01-01',
    end='2026-01-08',
    calendar_path='data/qlib/soxl/calendars/day.txt',
    data_paths=(str(CARD / 'daily_backtest.csv'),),
)
export = write_analysis_report(report, REPORT_DIR, dataset=dataset)
failure_md = write_failure_modes_markdown(failure_modes, CARD / 'failure_modes.md', title='SOXL 模型失败模式')
assert 'failure_modes' in export['paths']
assert not failure_modes.empty
failure_modes

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


,schema_version,model,baseline_model,segment_type,segment,failure_mode,severity_score,relative_return_pct_delta,stability_score_delta,max_drawdown_pct_delta,turnover_delta,diagnosis,review_note
0,analysis.failure_modes.v1,gcn_mse,ridge_mse,regime,stress,return_stability_lag,78.04619,-3.911862,-68.271389,3.902155,0.0,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。,gcn_mse 在 stress 同时落后基准 ridge_mse 的相对收益和稳定性，优先...
1,analysis.failure_modes.v1,gcn_mse,ridge_mse,window,stress_window,return_stability_lag,78.04619,-3.911862,-68.271389,3.902155,0.0,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。,gcn_mse 在 stress_window 同时落后基准 ridge_mse 的相对收益...
2,analysis.failure_modes.v1,gcn_mse,ridge_mse,window_regime,stress_window::stress,return_stability_lag,78.04619,-3.911862,-68.271389,3.902155,0.0,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。,gcn_mse 在 stress_window::stress 同时落后基准 ridge_m...
3,analysis.failure_modes.v1,gcn_mse,ridge_mse,all,all,return_stability_lag,45.16536,-3.000000,-36.013651,6.303419,0.0,弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。,gcn_mse 在 all 同时落后基准 ridge_mse 的相对收益和稳定性，优先检查该...


## 关键 failure mode

重点看 stress window：如果候选模型在压力段相对收益和稳定性同时弱于基准，不应仅因 bull 段收益更高而晋升。

In [4]:
top_failure = failure_modes.iloc[0].to_dict()
assert 'stress' in top_failure['segment']
assert top_failure['failure_mode'] == 'return_stability_lag'
assert 'predictions' not in json.loads((REPORT_DIR / 'analysis_manifest.json').read_text(encoding='utf-8')).get('paths', {})
top_failure

{'schema_version': 'analysis.failure_modes.v1',
 'model': 'gcn_mse',
 'baseline_model': 'ridge_mse',
 'segment_type': 'regime',
 'segment': 'stress',
 'failure_mode': 'return_stability_lag',
 'severity_score': 78.04619,
 'relative_return_pct_delta': -3.9118617744127393,
 'stability_score_delta': -68.27138855380323,
 'max_drawdown_pct_delta': 3.90215492137449,
 'turnover_delta': 0.0,
 'diagnosis': '弱于基准：分段相对收益和稳定性同时落后，需要检查信号、仓位或 regime 适配。',
 'review_note': 'gcn_mse 在 stress 同时落后基准 ridge_mse 的相对收益和稳定性，优先检查该 regime/window 的信号适配。'}

In [5]:
summary = {
    'schema_version': 'research.analysis_failure_modes_v1',
    'status': 'completed',
    'report_dir': str(REPORT_DIR.relative_to(REPO_ROOT)),
    'failure_modes_path': str((REPORT_DIR / 'failure_modes.csv').relative_to(REPO_ROOT)),
    'failure_modes_markdown_path': str(failure_md.relative_to(REPO_ROOT)),
    'failure_mode_count': int(len(failure_modes)),
    'top_failure_mode': str(top_failure['failure_mode']),
    'top_failure_segment': str(top_failure['segment']),
    'top_failure_model': str(top_failure['model']),
    'manifest_hash': json.loads((REPORT_DIR / 'analysis_manifest.json').read_text(encoding='utf-8'))['manifest_hash'],
    'conclusion': 'failure_modes.csv 和 failure_modes.md 已从 model_effect_deltas 派生，优先暴露 stress window 下的收益/稳定性同步落后。',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary

{'schema_version': 'research.analysis_failure_modes_v1',
 'status': 'completed',
 'report_dir': 'research/analysis_failure_modes_v1/analysis_report',
 'failure_modes_path': 'research/analysis_failure_modes_v1/analysis_report/failure_modes.csv',
 'failure_modes_markdown_path': 'research/analysis_failure_modes_v1/failure_modes.md',
 'failure_mode_count': 4,
 'top_failure_mode': 'return_stability_lag',
 'top_failure_segment': 'stress',
 'top_failure_model': 'gcn_mse',
 'manifest_hash': '6efbc09317a03a4046210278f6dad819cbe6fc25e0ec13b752e89d62c06dabd6',
 'conclusion': 'failure_modes.csv 和 failure_modes.md 已从 model_effect_deltas 派生，优先暴露 stress window 下的收益/稳定性同步落后。'}

## 中文结论

- 做了什么：新增并验证模型 failure mode 诊断，把 `model_effect_deltas` 归纳成 `failure_modes.csv` 和中文 `failure_modes.md`。
- 核心指标：本次识别出多个失败模式，最高优先级是 `gcn_mse` 在 stress window 的 `return_stability_lag`。
- regime 差异：候选模型在 bull 段收益更强，但 stress/stress_window 下相对收益、稳定性和回撤弱于 ridge 基准。
- 是否晋升：这是 analysis 框架能力，可保留在 `src/analysis`；示例模型不因 bull 表现更强而晋升。
- 风险：failure mode 阈值是稳定接口默认值，真实实验仍需结合交易成本、样本量和 qlib risk metrics 复核。
- 下一步：dashboard 可进一步把 `failure_modes` 做成单独对比面板，schedule 通知可带 top failure mode。